In [1]:
from pathlib import Path
pdf_paths = sorted(Path("../data/raw").glob("*.pdf"))
print(pdf_paths)

[WindowsPath('../data/raw/potato_disease-pest-management_cpri_2016_manual.pdf'), WindowsPath('../data/raw/rice_weed-mgmt-dry-direct-seeded_nrri_2019_brief.pdf'), WindowsPath('../data/raw/rice_weedy-rice-options_dwsr_2013_article.pdf'), WindowsPath('../data/raw/wheat_rust-resistant-varieties-economics_vpkas_2020_article.pdf')]


In [2]:

# from langchain_community.document_loaders import PyPDFLoader

# def meta_from_name(path):
#     crop, topic, org, year, kind = path.stem.split("_")
#     return {"doc_id": path.stem, "crop": crop, "topic": topic,
#             "org": org, "year": int(year), "kind": kind}

# pages = []
# for p in pdf_paths:
#     docs = PyPDFLoader(str(p)).load()
#     for d in docs:
#         d.metadata.update(meta_from_name(p))
#     print(p.name, len(docs), "pages")
#     pages.extend(docs)

In [3]:
import pymupdf
from langchain_core.documents import Document

KINDS = {"brief", "bulletin", "manual", "article", "guideline", "leaflet"}

def meta_from_name(path):
    crop, topic, org, year, kind = path.stem.split("_")
    assert kind in KINDS, f"unexpected kind {kind!r} in {path.name}"
    return {"doc_id": path.stem, "crop": crop, "topic": topic,
            "org": org, "year": int(year), "kind": kind}

def load_pdf(path):
    docs = []
    with pymupdf.open(str(path)) as pdf:
        for i, page in enumerate(pdf):
            meta = meta_from_name(path)
            meta.update({"source": path.name, "page": i + 1})
            docs.append(Document(page_content=page.get_text(), metadata=meta))
    return docs

pages = []
for p in pdf_paths:
    docs = load_pdf(p)
    print(p.name, len(docs), "pages")
    pages.extend(docs)

potato_disease-pest-management_cpri_2016_manual.pdf 142 pages
rice_weed-mgmt-dry-direct-seeded_nrri_2019_brief.pdf 8 pages
rice_weedy-rice-options_dwsr_2013_article.pdf 6 pages
wheat_rust-resistant-varieties-economics_vpkas_2020_article.pdf 8 pages


In [4]:
print(pages[5].metadata)
print(pages[5].page_content[:1000])

empty = [d for d in pages if len(d.page_content.strip()) < 50]
print(len(empty), "of", len(pages), "pages have almost no text")

{'doc_id': 'potato_disease-pest-management_cpri_2016_manual', 'crop': 'potato', 'topic': 'disease-pest-management', 'org': 'cpri', 'year': 2016, 'kind': 'manual', 'source': 'potato_disease-pest-management_cpri_2016_manual.pdf', 'page': 6}
CONTENTS
Chapter
No.
Title
Contributors
Pages
13.
Nematodes and their management
ARTI BAIRWA, VENKATASALM E P,
PRIYANK H M & SUDHA
ICAR-C.P.R.I, Shimla – 171 001
68 – 74
14.
Pesticide resistance management
GAURAV VERMA
Scientist
Division of Plant Protection
ICAR-C.P.R.I, Shimla – 171 001
75 – 88
15.
Disease forecasting for early and late blight of
potato
SANJEEV SHARMA
Senior Scientist
Division of Plant Protection
ICAR-C.P.R.I, Shimla – 171 001
89 – 92
16.
Forecasting of insect vectors for management
of potato virus diseases
SRIDHAR J, V VENKATESWARLU,
KAMLESH MALIK, ANUJ
BHATNAGAR & M. ABAS SHAH
Division of Plant Protection
ICAR-C.P.R.I, Shimla – 171 001
93 – 98
17.
Role of nutrients in potato health
JAGDEV SHARMA
Principal Scientist
Division of Crop

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import math
from collections import Counter
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(pages)

actual = Counter((c.metadata["doc_id"], c.metadata["page"]) for c in chunks)
diffs = []
for d in pages:
    key = (d.metadata["doc_id"], d.metadata["page"])
    L = len(d.page_content)
    pred = 1 if L <= 1000 else 1 + math.ceil((L - 1000) / 900)
    diffs.append((actual[key] - pred, key, L))

print(Counter(x[0] for x in diffs))
print(sorted(diffs)[:5])

Counter({0: 160, -1: 4})
[(-1, ('potato_disease-pest-management_cpri_2016_manual', 17), 4643), (-1, ('potato_disease-pest-management_cpri_2016_manual', 60), 4621), (-1, ('potato_disease-pest-management_cpri_2016_manual', 95), 3720), (-1, ('potato_disease-pest-management_cpri_2016_manual', 100), 4697), (0, ('potato_disease-pest-management_cpri_2016_manual', 1), 169)]


In [6]:
from collections import defaultdict

by_doc = defaultdict(list)
for d in pages:
    by_doc[d.metadata["doc_id"]].append(d)

for doc_id, doc_pages in by_doc.items():
    counts = Counter(l.strip() for d in doc_pages
                     for l in set(d.page_content.splitlines()) if l.strip())
    cutoff = max(5, 0.5 * len(doc_pages))
    boiler = {l for l, n in counts.items() if n >= cutoff and len(l) >= 10}
    for d in doc_pages:
        text = d.page_content.replace("\uf0b7", "•")
        d.page_content = "\n".join(l for l in text.splitlines() if l.strip() not in boiler)

In [7]:
import math
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_stats(pages, size, overlap):
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    chunks = splitter.split_documents(pages)
    lens = [len(c.page_content) for c in chunks]
    print(size, overlap, "chunks:", len(chunks),
          "min/avg/max:", min(lens), sum(lens)//len(lens), max(lens))
    return chunks

def predicted(pages, size, overlap):
    total = 0
    for d in pages:
        L = len(d.page_content)
        if L == 0:
            continue
        total += 1 if L <= size else 1 + math.ceil((L - size) / (size - overlap))
    return total

for size, overlap in [(500, 50), (1000, 100), (1500, 150), (1000, 0)]:
    chunk_stats(pages, size, overlap)
    print("   predicted by formula:", predicted(pages, size, overlap))

500 50 chunks: 1176 min/avg/max: 51 424 500
   predicted by formula: 1171
1000 100 chunks: 615 min/avg/max: 81 829 1000
   predicted by formula: 620
1500 150 chunks: 432 min/avg/max: 93 1212 1500
   predicted by formula: 432
1000 0 chunks: 600 min/avg/max: 15 824 1000
   predicted by formula: 578


In [8]:
chunks = chunk_stats(pages, 1000, 100)
import random
for c in random.sample(chunks, 3):
    print(c.metadata)
    print(c.page_content)
    print("-" * 60)

1000 100 chunks: 615 min/avg/max: 81 829 1000
{'doc_id': 'wheat_rust-resistant-varieties-economics_vpkas_2020_article', 'crop': 'wheat', 'topic': 'rust-resistant-varieties-economics', 'org': 'vpkas', 'year': 2020, 'kind': 'article', 'source': 'wheat_rust-resistant-varieties-economics_vpkas_2020_article.pdf', 'page': 4}
of improved variety of VL Gehun 907 along with 
recommended practices gave higher net return of₹Rs. 
26,671 on farmers' fields indicated that improved wheat 
production technologies under FLD of VL Gehun 953 
produced an average yield of 3360, 3130 and 3760 kg/ha 
during 2016-17, 2017-18 and 2018-19 respectively, which 
were 34.9 and ₹54,950 compared to₹12,388 and₹39,972 
per hectare under farmers practice during 2014-15 and 
2016-17, respectively. The profitability analysis exhibited 
that cultivation of VL Gehun 907 was more profitable 
with benefit cost ratio of 2.06 (2014-15) and 3.02 (2016-17).
------------------------------------------------------------
{'doc_id': 

In [9]:
from collections import Counter
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(pages)

seen = Counter()
ids = []
for c in chunks:
    key = (c.metadata["doc_id"], c.metadata["page"])
    ids.append(f"{key[0]}:p{key[1]}:c{seen[key]}")
    seen[key] += 1

assert len(set(ids)) == len(ids)
print(len(chunks), ids[:2])

615 ['potato_disease-pest-management_cpri_2016_manual:p1:c0', 'potato_disease-pest-management_cpri_2016_manual:p2:c0']


## Don't run the following cell again 

In [ ]:
#Warning - Don't run this twice (Not needed and takes a lot of computation )
import time
import numpy as np
from langchain_ollama import OllamaEmbeddings

emb = OllamaEmbeddings(model="nomic-embed-text")
texts = [c.page_content for c in chunks]

vectors, t0 = [], time.time()
for i in range(0, len(texts), 32):
    vectors.extend(emb.embed_documents(texts[i:i + 32]))
    print(min(i + 32, len(texts)), "of", len(texts), round(time.time() - t0), "s")
V = np.array(vectors)
print(V.shape)

32 of 615 25 s
64 of 615 44 s
96 of 615 65 s
128 of 615 89 s
160 of 615 112 s
192 of 615 137 s
224 of 615 160 s
256 of 615 184 s
288 of 615 206 s
320 of 615 225 s
352 of 615 245 s
384 of 615 265 s
416 of 615 286 s
448 of 615 307 s
480 of 615 326 s
512 of 615 348 s
544 of 615 370 s
576 of 615 395 s
608 of 615 420 s
615 of 615 425 s
(615, 768)


In [11]:
import chromadb
from chromadb.config import Settings

client = chromadb.PersistentClient(
    path="../data/chroma/mitra",
    settings=Settings(anonymized_telemetry=False)
)
col = client.get_or_create_collection(
    "mitra_chunks",
    configuration={"hnsw": {"space": "cosine"}},
    metadata={
        "embed_model": "nomic-embed-text",
        "dim": int(V.shape[1]),
        "chunk_size": 1000,
        "chunk_overlap": 100,
        "doc_prefix": "none"
    },
)
col.upsert(
    ids=ids,
    embeddings=V.tolist(),
    documents=texts,
    metadatas=[c.metadata for c in chunks]
)
print(col.count())

615


In [12]:
def search(question, k=3, crop=None):
    qv = emb.embed_query(question)
    res = col.query(
        query_embeddings=[qv],
        n_results=k,
        where={"crop": crop} if crop else None
    )
    for dist, meta, doc in zip(res["distances"][0], res["metadatas"][0], res["documents"][0]):
        print(round(1 - dist, 3), meta["doc_id"], "p", meta["page"])
        print("    ", doc[:200].replace("\n", " "))

In [1]:
import chromadb
from chromadb.config import Settings

client = chromadb.PersistentClient(
    path="../data/chroma/mitra",
    settings=Settings(anonymized_telemetry=False)
)
col = client.get_collection("mitra_chunks")
print(col.count(), col.metadata)

615 {'chunk_overlap': 100, 'embed_model': 'nomic-embed-text', 'doc_prefix': 'none', 'chunk_size': 1000, 'dim': 768}
